# Team Colab runner — setup, preprocess, train, evaluate

End-to-end training runner used by the team for trajectory-representation experiments. Handles Drive mount, repo clone, nuPlan extraction, data preprocessing, training, evaluation, and artifact archival.

**Setup requirements**: see `QUICKSTART.md` at the repo root.

## 1. Config

**This is the only cell you edit per run.** Set `KINEMATIC` (`waypoints` or `frenet`), `SEED`, and the Drive folder. Everything else is derived. To force re-preprocessing, flip the optional `WIPE_*` flags at the bottom.

In [ ]:
# Edit only if you want non-default values
DRIVE_FOLDER         = '/content/drive/MyDrive/cs269'
RUN_SEED             = 269   # v7: paired retrain at seed 269 (was 42 in v6)

# ===== Per-session run knobs (THE knobs teammates edit) =====
# RUN_KINEMATIC: which model to train + eval in this session.
#   'frenet'    — Frenet (s, d, cos_h, sin_h) target with Option A centerline conditioning
#   'waypoints' — Cartesian (x, y) waypoint target (the v6 baseline / paper architecture)
# Run TWICE per session: once with 'frenet', restart runtime, once with 'waypoints'.
RUN_KINEMATIC        = 'frenet'   # 'frenet' | 'waypoints'
V7_VARIANT           = 'AUDIT'    # 'AUDIT' | 'h' (scale-up) — usually leave on AUDIT
assert RUN_KINEMATIC in {'frenet', 'waypoints'}, f'RUN_KINEMATIC={RUN_KINEMATIC!r}'
assert V7_VARIANT in {'AUDIT', 'h', 'b', 'c', 'd'}, f'V7_VARIANT={V7_VARIANT!r}'

TOTAL_SCENARIOS      = 1500
assert TOTAL_SCENARIOS is None or TOTAL_SCENARIOS >= 1000, (
    f'TOTAL_SCENARIOS={TOTAL_SCENARIOS}: must be None (unlimited) or >= 1000. '
    'A cap below 1000 produces a degenerate training set. '
    'A value of 0 is ambiguous across nuPlan versions; use None instead.'
)

TRAIN_EPOCHS         = 50
TRAIN_BATCH_SIZE     = 32

# Held-out test set (best-effort - skipped if raw nuPlan data not available).
# After the Phase 2 methodology rewrite, log-disjoint train/val split is
# enforced via --log_names_json instead of an offset-based skip. The
# HELDOUT_SCENARIOS value here is a safety cap on the val-log preprocess;
# the actual val cache size depends on how many scenarios live in the val logs.
#
# Scale-up plan: when running the full mini-split (TOTAL_SCENARIOS=None),
# bump the held-out cap proportionally so the eval signal stays
# statistically meaningful. The val logs hold ~1500-2000 scenarios total.
HELDOUT_SCENARIOS    = 300 if (TOTAL_SCENARIOS is not None and TOTAL_SCENARIOS <= 2000) else 1500

WARM_UP_EPOCHS       = min(5, max(0, TRAIN_EPOCHS - 1))
EFFECTIVE_BATCH_SIZE = min(TRAIN_BATCH_SIZE, TOTAL_SCENARIOS) if TOTAL_SCENARIOS else TRAIN_BATCH_SIZE

assert TRAIN_EPOCHS == 0 or WARM_UP_EPOCHS < TRAIN_EPOCHS, (
    f'WARM_UP_EPOCHS ({WARM_UP_EPOCHS}) must be < TRAIN_EPOCHS ({TRAIN_EPOCHS})'
)

# Drive paths
DRIVE_ZIPS              = f'{DRIVE_FOLDER}/nuplan_zips'
DRIVE_CKPT              = f'{DRIVE_FOLDER}/checkpoints'
DRIVE_RESULTS           = f'{DRIVE_FOLDER}/results'
DRIVE_REQUIREMENTS      = f'{DRIVE_FOLDER}/working_requirements.txt'
# Scale-aware Drive cache folder: when running at full scale we point at a
# distinct Drive folder so the smaller 1500-scenario cache is not clobbered.
DRIVE_PREPROCESSED_1500 = f'{DRIVE_FOLDER}/preprocessed_cache_1500'
DRIVE_PREPROCESSED_FULL = f'{DRIVE_FOLDER}/preprocessed_cache_full'
DRIVE_PREPROCESSED      = DRIVE_PREPROCESSED_FULL if TOTAL_SCENARIOS is None else DRIVE_PREPROCESSED_1500
DRIVE_HELDOUT           = f'{DRIVE_FOLDER}/heldout_cache_{HELDOUT_SCENARIOS}'

# Local paths
LOCAL_ROOT           = '/content/work'
LOCAL_NUPLAN         = f'{LOCAL_ROOT}/nuplan'
LOCAL_MAPS           = f'{LOCAL_NUPLAN}/maps'
LOCAL_LOGS           = f'{LOCAL_NUPLAN}/data/cache/mini'
LOCAL_EXP            = f'{LOCAL_NUPLAN}/exp'
LOCAL_CACHE          = f'{LOCAL_ROOT}/preprocessed_cache'
LOCAL_HELDOUT_CACHE  = f'{LOCAL_ROOT}/heldout_cache'
LOCAL_RUNS           = f'{LOCAL_ROOT}/runs'
LOCAL_TB             = f'{LOCAL_ROOT}/tensorboard'

# Code / env paths
REPO_DIR             = '/content/CS269FlowPlannerProject'
FP_DIR               = f'{REPO_DIR}/flow_planner'
NUPLAN_DEVKIT_DIR    = '/content/nuplan-devkit'
VENV                 = '/content/venv39'
PYTHON               = f'{VENV}/bin/python'
PIP                  = f'{VENV}/bin/pip'
CONSTRAINTS          = f'{LOCAL_ROOT}/pip_constraints.txt'  # live under LOCAL_ROOT so /tmp wipe doesn't lose it

# Methodology: log-disjoint train/val split (Phase 2 deliverable).
# Split seed intentionally pinned to 42 so v6 / v7 train on the SAME logs -
# only RUN_SEED (model init) varies between runs. Do NOT roll this to 269.
# See scripts/generate_log_split.py and docs/preprocessing_methodology.md s4.
# If docs/log_split_mini_seed42.json is not committed to the repo, cells 31
# and 41 generate it lazily once LOCAL_LOGS is populated.
LOG_SPLIT_JSON       = f'{REPO_DIR}/docs/log_split_mini_seed42.json'

# Tracks whether held-out eval is available this session (set later)
HELDOUT_AVAILABLE = False

# Sentinel so later cells can assert this cell ran (e.g. after kernel restart)
_CONFIG_CELL_RAN = True

_total_label = 'unlimited (full mini-split)' if TOTAL_SCENARIOS is None else str(TOTAL_SCENARIOS)
print(f'Train scenarios cap: {_total_label}, held-out cap: {HELDOUT_SCENARIOS}')
print(f'Epochs: {TRAIN_EPOCHS} (warmup {WARM_UP_EPOCHS}), batch: {EFFECTIVE_BATCH_SIZE}')
print(f'Drive cache: {DRIVE_PREPROCESSED}')
print(f'Log split: {LOG_SPLIT_JSON} (seed 42 pinned; RUN_SEED={RUN_SEED} is model-init only)')

# ---- Optional: force re-preprocessing by uncommenting one or both flags ----
# Optional: wipe preprocessed caches before this run.
#
# WIPE_LOCAL_CACHE  is cheap — just /content/work; survives any later cell.
# WIPE_DRIVE_CACHE  is EXPENSIVE — destroys preprocessed_cache_1500 on Drive,
#                   triggering a ~30-60 minute rebuild via Section 7b.
#
# Default both False. FORCE_REPREPROCESS kept as a legacy alias that turns
# ON the Drive wipe (cell 12 honors it for backward compatibility) so older
# muscle memory still works, but new sessions should prefer the explicit
# WIPE_* flags.
# WIPE_LOCAL_CACHE   = False
# WIPE_DRIVE_CACHE   = False
# FORCE_REPREPROCESS = False  # legacy alias: True implies WIPE_DRIVE_CACHE=True
# if FORCE_REPREPROCESS:
    # WIPE_DRIVE_CACHE = True
# print(f'WIPE_LOCAL_CACHE={WIPE_LOCAL_CACHE}, WIPE_DRIVE_CACHE={WIPE_DRIVE_CACHE} '
      # f'(actual wipe happens in cell 12 after Drive mount + config)')


## 2. Setup

Mount Google Drive, clone the team repo (PAT from Colab Secrets), build a Python 3.9 venv, and install dependencies. The fast path reuses a captured `working_requirements.txt` from Drive; a sanity check runs before training.

In [ ]:
# Force-fresh clone with PAT  clears the broken repo and pulls main fresh.
import subprocess, shutil, pathlib, os
from google.colab import userdata

REPO_PATH = '/content/CS269FlowPlannerProject'
PAT = userdata.get('token')  # Colab secret named 'token' (same one cell 12 uses)
assert PAT, "Colab secret 'token' is missing. Set it in the left sidebar 'Secrets' panel."

if pathlib.Path(REPO_PATH).exists():
    shutil.rmtree(REPO_PATH)

# Build the authenticated URL but never print it. Use git credential.helper
# so the PAT isn't on argv and isn't written to the repo's .git/config.
url_with_pat = f'https://{PAT}@github.com/wimaan3/CS269FlowPlannerProject.git'
try:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', url_with_pat, REPO_PATH],
        check=True, capture_output=True, text=True,
    )
finally:
    # Scrub the URL from local memory ASAP
    del url_with_pat

# Reset origin to the no-PAT URL so the PAT doesn't end up in .git/config
subprocess.run(
    ['git', '-C', REPO_PATH, 'remote', 'set-url', 'origin',
     'https://github.com/wimaan3/CS269FlowPlannerProject.git'],
    check=True,
)
subprocess.run(['git', '-C', REPO_PATH, 'log', '-1', '--oneline'], check=True)
print('repo re-cloned fresh on main')


In [ ]:
from google.colab import drive
import pathlib, time

assert globals().get('_CONFIG_CELL_RAN'), 'Run the Config cell (Section 1) before this cell. _CONFIG_CELL_RAN is unset.'

# Idempotent mount — if already mounted, don't re-prompt
if not pathlib.Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('Drive already mounted at /content/drive')

# Brief FUSE-lag retry — mount can return before subfolders are listable
for attempt in range(10):
    if pathlib.Path(DRIVE_FOLDER).exists():
        break
    time.sleep(1)

assert pathlib.Path(DRIVE_FOLDER).exists(), (
    f'DRIVE_FOLDER {DRIVE_FOLDER} not visible after mount. '
    f'Check that the folder name in cell 6 matches your Drive (case-sensitive).'
)

print(f'\nDrive folder: {DRIVE_FOLDER}')
for sub in ['nuplan_zips', 'preprocessed_cache_1500', 'checkpoints', 'experiments', 'results', 'heldout_cache']:
    p = pathlib.Path(f'{DRIVE_FOLDER}/{sub}')
    if p.exists():
        n = sum(1 for _ in p.iterdir())
        print(f'  {sub}/  ({n} items)')
    else:
        print(f'  {sub}/  MISSING')
print()
req_path = pathlib.Path(DRIVE_REQUIREMENTS)
print(f'  working_requirements.txt  {"present" if req_path.exists() else "MISSING"}')


In [ ]:
import pathlib, subprocess, shutil

def _sh(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
        raise RuntimeError(f'Command failed: {cmd[:4]}...')
    return r

# Validate existing venv before reusing it (catches corrupted half-installs)
venv_ok = False
if pathlib.Path(VENV).exists():
    probe = subprocess.run([PYTHON, '--version'], capture_output=True, text=True)
    venv_ok = (probe.returncode == 0 and '3.9' in probe.stdout)
    if not venv_ok:
        print(f'Existing venv at {VENV} is broken or wrong version — rebuilding')
        shutil.rmtree(VENV, ignore_errors=True)

if not venv_ok:
    print('Installing python3.9 (deadsnakes PPA + apt)...')
    # python3.9 is NOT in Ubuntu 22.04 default sources — need deadsnakes PPA.
    _sh(['sudo', 'add-apt-repository', '-y', 'ppa:deadsnakes/ppa'])
    _sh(['sudo', 'apt-get', 'update', '-qq'])
    _sh(['sudo', 'apt-get', 'install', '-y', 'python3.9', 'python3.9-venv', 'python3.9-dev'])
    # Sanity: python3.9 binary must actually exist now
    py39_probe = subprocess.run(['python3.9', '--version'], capture_output=True, text=True)
    assert py39_probe.returncode == 0, f'python3.9 still not installed after apt: {py39_probe.stderr}'
    print(f'  apt OK: {py39_probe.stdout.strip()}')
    _sh(['python3.9', '-m', 'venv', VENV])
    _sh([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel', '-q'])
else:
    print(f'venv exists and is valid at {VENV}')

ver = subprocess.run([PYTHON, '--version'], capture_output=True, text=True).stdout.strip()
assert '3.9' in ver, f'Wrong Python version in venv: {ver} (expected 3.9.x)'
print(ver)


In [ ]:
# Clone nuplan-devkit (required for the editable install in the next cell)
import pathlib, subprocess
if not pathlib.Path(NUPLAN_DEVKIT_DIR).exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/motional/nuplan-devkit.git',
         NUPLAN_DEVKIT_DIR],
        check=True,
    )
    print(f'cloned nuplan-devkit -> {NUPLAN_DEVKIT_DIR}')
else:
    print(f'nuplan-devkit already present at {NUPLAN_DEVKIT_DIR}')

In [ ]:
import pathlib, time

# FUSE-lag retry: DRIVE_REQUIREMENTS may briefly read as missing right after
# mount even when present on Drive.
for _ in range(5):
    if pathlib.Path(DRIVE_REQUIREMENTS).exists():
        break
    time.sleep(1)

FAST = pathlib.Path(DRIVE_REQUIREMENTS).exists()
if FAST:
    # Probe size — empty file from a failed prior capture should NOT trigger FAST
    size = pathlib.Path(DRIVE_REQUIREMENTS).stat().st_size
    if size < 100:
        print(f'  {DRIVE_REQUIREMENTS} is suspiciously small ({size} bytes) — forcing SLOW path')
        FAST = False

print(f'Install path: {"FAST" if FAST else "SLOW"}')
if FAST:
    print(f'  Using cached requirements from {DRIVE_REQUIREMENTS}')
else:
    print('  Will install from scratch (~15-20 min)')

# Constraints file pinning numpy<2 (must persist across all pip invocations).
# Live under LOCAL_ROOT, not /tmp (Colab wipes /tmp between reconnects).
pathlib.Path(LOCAL_ROOT).mkdir(parents=True, exist_ok=True)
with open(CONSTRAINTS, 'w') as f:
    f.write('# pip constraints — pins numpy<2 for nuplan/flow_planner compatibility\nnumpy<2\n')

# FAST path: install from captured working_requirements.txt with --no-deps,
# then run `pip check` to catch missing transitive deps loudly.
# SLOW path: install nuplan-devkit requirements with filtering for broken RC pins.
import pathlib, subprocess

def _pip(args, **kw):
    r = subprocess.run([PIP] + list(args), capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print('STDOUT (tail):', '\n'.join(r.stdout.splitlines()[-25:]))
        print('STDERR (tail):', '\n'.join(r.stderr.splitlines()[-25:]))
        raise RuntimeError(f'pip failed: {args[:2]}')
    # On success print just the last few lines for visibility
    tail = '\n'.join(r.stdout.splitlines()[-10:])
    print(tail)
    return r

if FAST:
    # Strip editable lines (we redo those separately) and filter our own packages
    clean = pathlib.Path('/tmp/working_clean_reqs.txt')
    with open(DRIVE_REQUIREMENTS) as fin, clean.open('w') as fout:
        for line in fin:
            s = line.strip()
            if (not s or s.startswith('#') or s.startswith('-e ') or '@ file://' in s
                or 'flow_planner' in s or 'nuplan-devkit' in s or 'diffusion_planner' in s):
                continue
            fout.write(line)
    assert clean.stat().st_size > 100, f'Cleaned reqs file is empty/tiny: {clean}'
    _pip(['install', '-r', str(clean), '--no-deps'])
    # CRITICAL: --no-deps means we must verify the dep graph is satisfied
    chk = subprocess.run([PIP, 'check'], capture_output=True, text=True)
    if chk.returncode != 0:
        print('pip check found missing/broken deps (FAST path is incomplete):')
        print(chk.stdout)
        # Don't abort — but try to repair by installing missing deps with constraints
        print('Attempting to repair by installing flow_planner requirements with constraints...')
        _pip(['install', '-r', f'{FP_DIR}/requirements.txt', '-c', CONSTRAINTS])
        chk2 = subprocess.run([PIP, 'check'], capture_output=True, text=True)
        if chk2.returncode != 0:
            print('pip check STILL failing after repair:')
            print(chk2.stdout)
            print('You can proceed (sanity check in cell 21 will catch flow_planner import failures)')
            print('or delete DRIVE_REQUIREMENTS and re-run to get a fresh SLOW install.')
        else:
            print('Repair OK — dep graph now satisfied.')
    else:
        print('pip check OK — all deps satisfied.')
else:
    # SLOW path: filter nuplan-devkit requirements (drop broken hydra/omegaconf RC pins)
    req_in = f'{NUPLAN_DEVKIT_DIR}/requirements.txt'
    req_out = '/tmp/filtered_requirements.txt'
    bad_pins = ['hydra-core', 'omegaconf']
    with open(req_in) as f, open(req_out, 'w') as g:
        for line in f:
            # Strip leading whitespace before the prefix check so '  hydra-core==X' is caught
            stripped = line.lstrip().lower()
            if stripped.startswith('#'):
                g.write(line); continue
            if not any(stripped.startswith(b) for b in bad_pins):
                g.write(line)
    _pip(['install', '-r', req_out, '-c', CONSTRAINTS])
    _pip(['install', 'hydra-core>=1.2,<1.4', 'omegaconf>=2.2,<2.4', '-c', CONSTRAINTS])
    _pip(['install', '-r', f'{FP_DIR}/requirements.txt', '-c', CONSTRAINTS])

# Editable install of nuplan-devkit + flow_planner so our modifications take effect
!{PIP} install -e {NUPLAN_DEVKIT_DIR} --no-deps 2>&1 | tail -3
!{PIP} install -e {FP_DIR} --no-deps 2>&1 | tail -3

# Defense-in-depth: pin numpy<2
import subprocess
r = subprocess.run([PIP, 'install', 'numpy<2', '--force-reinstall', '--no-deps', '-q'],
                   capture_output=True, text=True)
if r.returncode != 0:
    print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
    raise RuntimeError('numpy<2 pin failed')
ver = subprocess.run([PYTHON, '-c', 'import numpy; print(f"numpy {numpy.__version__}")'],
                     capture_output=True, text=True).stdout.strip()
print(ver)
assert ver.startswith('numpy 1.'), f'Wrong numpy version: {ver} (expected 1.x)'


In [ ]:
# Implementation lives at scripts/check_sanity.py — runs in the venv's
# Python 3.9 because the notebook kernel (Colab 3.11) cannot import
# nuplan / flow_planner.
import os, subprocess

r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/check_sanity.py'],
    capture_output=True, text=True,
    env={**os.environ, 'FP_DIR': FP_DIR},
)
print(r.stdout)
all_ok = r.returncode == 0
if not all_ok:
    print('STDERR:'); print(r.stderr)
print(f'\nall_ok = {all_ok}')
assert all_ok, 'Sanity check failed — fix above before continuing'


In [ ]:
if all_ok and not FAST:
    print(f'Capturing {DRIVE_REQUIREMENTS}...')
    !{PIP} freeze > {DRIVE_REQUIREMENTS}
    print('Done')
else:
    print('Skipping (FAST path already used existing file)')


## 3. Data

Load the preprocessed `.npz` cache from Drive if present; otherwise unzip raw nuPlan and run `preprocess`. Then validate the cache and write `train_manifest.json`. A held-out cache is built from the val-log split when raw data is available — eval on held-out is skipped cleanly if not.

In [ ]:
import pathlib, time, subprocess, shutil

# Honour the scale-aware Drive cache picked in cell 6 (DRIVE_PREPROCESSED).
# Older sessions still wrote to preprocessed_cache_1500; keep the legacy var
# as a synonym for any older cell references downstream.
drive_cache = pathlib.Path(DRIVE_PREPROCESSED)
local_cache = pathlib.Path(LOCAL_CACHE)
local_cache.mkdir(parents=True, exist_ok=True)

# train_cache_ready signals to Section 7b whether the fallback unzip +
# preprocess should run. True after a successful Drive-cache load (even if
# methodology-warning fired); False if the Drive cache is missing.
train_cache_ready = False

if drive_cache.exists():
    drive_npz = sorted(drive_cache.glob('*.npz'))
    print(f'Drive cache: {len(drive_npz)} .npz files at {drive_cache}')

    if len(drive_npz) == 0:
        raise RuntimeError(f'Drive cache {drive_cache} exists but contains no .npz files')

    # Scale-up fix: replace the previous per-file `!cp` loop (~100-300ms/file
    # via FUSE + bash subprocess fork, i.e. tens of minutes at ~10k files)
    # with a single rsync call. rsync's --ignore-existing handles "what to
    # copy" in one pass, so we also skip the upfront 9999-call stat() scan.
    t = time.time()
    print(f'Copying via rsync (Drive -> local) ...')
    r = subprocess.run(
        ['rsync', '-a', '--ignore-existing', '--info=stats2',
         f'{drive_cache}/', f'{local_cache}/'],
        capture_output=True, text=True,
    )
    # rsync exit 0 = ok, 24 = "some files vanished" (benign for Drive FUSE).
    if r.returncode not in (0, 24):
        print(r.stdout); print(r.stderr)
        raise RuntimeError(f'rsync failed with exit code {r.returncode}')
    print(r.stdout.strip().splitlines()[-3:] if r.stdout else '')
    print(f'rsync done in {time.time()-t:.1f}s')

    # Also copy the preprocess_manifest.json (written next to the .npz files
    # by flow_planner/run_script/preprocess.py). The methodology-compliance
    # check below reads it from local_cache; without this copy the check
    # always fires the WARNING branch even when the cache was actually
    # methodology-compliant.
    drive_manifest = drive_cache / 'preprocess_manifest.json'
    if drive_manifest.exists():
        shutil.copy(str(drive_manifest), str(local_cache / 'preprocess_manifest.json'))
        print('Copied preprocess_manifest.json from Drive')

    # Post-copy integrity: catches truncated / zero-byte files. We compare
    # against Drive's per-file sizes only when the index is cheap; at full
    # scale we settle for zero-byte detection because the full stat scan is
    # itself expensive on Drive FUSE.
    local_npz_post = sorted(local_cache.glob('*.npz'))
    zero_byte = [p for p in local_npz_post if p.stat().st_size == 0]
    if zero_byte:
        raise RuntimeError(
            f'Post-copy integrity check failed: {len(zero_byte)} zero-byte files. '
            f'Delete those files locally and re-run this cell.'
        )
    print(f'Post-copy OK: {len(local_npz_post)} files in {local_cache}')

    # --- Methodology compliance check ---
    # The Drive cache may have been generated before the Phase 2 log-disjoint
    # split was introduced. If the manifest says it was NOT preprocessed with
    # --log_names_key=train pointing to LOG_SPLIT_JSON, we warn loudly (but
    # do not delete or auto-regenerate - the user decides).
    import json as _json
    manifest_path = local_cache / 'preprocess_manifest.json'
    methodology_ok = False
    if manifest_path.exists():
        _m = _json.loads(manifest_path.read_text())
        _lnj = _m.get('log_names_json')
        _lnk = _m.get('log_names_key')
        if _lnj and _lnk == 'train':
            methodology_ok = True
            print(f'Methodology: cache built with log_names_json={_lnj} key={_lnk!r} OK')

    if not methodology_ok:
        print()
        print('!!!!!! METHODOLOGY WARNING !!!!!!')
        if manifest_path.exists():
            print(f'!!! preprocess_manifest.json present but log_names_key != "train"')
            print(f'!!! manifest log_names_json: {_m.get("log_names_json")!r}')
            print(f'!!! manifest log_names_key:  {_m.get("log_names_key")!r}')
        else:
            print('!!! No preprocess_manifest.json in cache - likely pre-methodology cache.')
        print('!!! This cache may contain scenarios from val logs (DATA LEAKAGE).')
        print('!!! For methodology-compliant training:')
        print(f'!!!   1. Delete the Drive cache {DRIVE_PREPROCESSED}')
        print('!!!   2. Re-run cell 22 (will fall through to Section 7b which uses')
        print('!!!      --log_names_key=train and produces a clean train cache)')
        print('!!! Or proceed with the existing cache and disclose the')
        print('!!! limitation in the report.')
        print()

    # Cache was loaded from Drive (regardless of methodology status - the user
    # has been warned and can decide). Section 7b will skip.
    train_cache_ready = True
else:
    print(f'Drive cache {drive_cache} NOT found - will fall back to unzip + preprocess')
    print('(Section 7b - cells 27 and 28 - will run; Section 7c will validate)')

# Fallback path: unzip nuPlan from Drive. Runs only if cell 22 could not load
# the Drive-cache shortcut. Uses an explicit filename → target mapping rather
# than substring matching so a future Drive zip named "mini-with-maps.zip" or
# similar can\'t silently route to the wrong directory.
if not train_cache_ready:
    import time, pathlib, zipfile

    ZIP_TARGETS = {
        'nuplan-maps-v1.0.zip': LOCAL_MAPS,
        'nuplan-v1.1_mini.zip': LOCAL_LOGS,
    }

    pathlib.Path(LOCAL_MAPS).mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_LOGS).mkdir(parents=True, exist_ok=True)
    local_zip_dir = pathlib.Path(f'{LOCAL_ROOT}/_zips')
    local_zip_dir.mkdir(parents=True, exist_ok=True)

    for z in sorted(pathlib.Path(DRIVE_ZIPS).glob('*.zip')):
        if z.name not in ZIP_TARGETS:
            print(f'  SKIP {z.name}: not in ZIP_TARGETS allowlist')
            continue
        target = ZIP_TARGETS[z.name]
        # If target is already populated (previous extraction completed),
        # skip the unzip step — saves several minutes on re-runs.
        if any(pathlib.Path(target).iterdir()):
            print(f'  {z.name}: {target} already populated, skipping extract')
            continue
        dest = local_zip_dir / z.name
        if not dest.exists():
            t = time.time()
            # rsync is more reliable than shutil.copy for large Drive files
            !rsync --inplace {z} {dest}
            print(f'Copied {z.name} in {time.time()-t:.1f}s')
        try:
            with zipfile.ZipFile(dest) as zf:
                print(f'Unzipping {z.name} -> {target} ...')
                t = time.time()
                zf.extractall(target)
                print(f'  done in {time.time()-t:.1f}s')
        except zipfile.BadZipFile:
            print(f'  ERROR: {dest} is corrupted. Try re-uploading to Drive.')
else:
    print('Drive cache loaded successfully in cell 22 — fallback unzip not needed.')


In [ ]:
# AUTO-FLATTEN: nuPlan zips often extract into nested dirs (e.g.
# LOCAL_MAPS/maps/... instead of LOCAL_MAPS/...). The preprocess step in
# the next cells needs the CANONICAL paths, so we detect the nested
# structure once and move it up. Runs unconditionally — if the structure
# is already flat or the Drive-cache fast-path was used, it's a no-op.
import pathlib, shutil

def _flatten_if_nested(parent: pathlib.Path, expected_marker: str):
    """If parent is empty / lacks expected_marker but parent/<subdir>/expected_marker
    exists, move parent/<subdir>/* up into parent."""
    if not parent.exists():
        return False
    if (parent / expected_marker).exists():
        return False  # already flat
    # Look one level deep for the marker
    for sub in parent.iterdir():
        if sub.is_dir() and (sub / expected_marker).exists():
            print(f'  flattening {sub} -> {parent}')
            for item in list(sub.iterdir()):
                target = parent / item.name
                if target.exists():
                    continue  # don't clobber
                shutil.move(str(item), str(target))
            try:
                sub.rmdir()
            except OSError:
                pass
            return True
    return False

# Maps: expect LOCAL_MAPS/nuplan-maps-v1.0.json directly
maps_p = pathlib.Path(LOCAL_MAPS)
if _flatten_if_nested(maps_p, 'nuplan-maps-v1.0.json'):
    print(f'  flattened maps; now have: {sorted(p.name for p in maps_p.iterdir())[:5]}')

# Logs: expect LOCAL_LOGS/*.db directly. Walk two levels deep because zips
# sometimes nest as logs/data/cache/mini/*.db.
logs_p = pathlib.Path(LOCAL_LOGS)
if logs_p.exists() and not any(logs_p.glob('*.db')):
    candidates = list(logs_p.rglob('*.db'))
    if candidates:
        first_dir = candidates[0].parent
        print(f'  flattening logs from {first_dir} -> {logs_p}')
        for db in candidates:
            target = logs_p / db.name
            if not target.exists():
                shutil.move(str(db), str(target))
        # Best-effort cleanup of empty intermediate dirs
        for sub in sorted([p for p in logs_p.rglob('*') if p.is_dir()], key=lambda p: -len(p.parts)):
            try: sub.rmdir()
            except OSError: pass

print(f'After flatten:')
print(f'  {LOCAL_MAPS}/nuplan-maps-v1.0.json exists: {(maps_p/"nuplan-maps-v1.0.json").exists()}')
print(f'  {LOCAL_LOGS}/*.db count: {len(list(logs_p.glob("*.db")))}')

# the chronic 'preprocess produced 0 .npz files, then validate fails
# with cryptic message' failure mode.
import pathlib as _p
if not train_cache_ready:
    _maps = _p.Path(f'{LOCAL_MAPS}/nuplan-maps-v1.0.json')
    _dbs  = list(_p.Path(LOCAL_LOGS).glob('*.db'))
    assert _maps.exists(), (
        f'Cannot run preprocess: maps file missing at {_maps}. '
        f'Either the extract+flatten cells did not run, or '
        f'nuplan-maps-v1.0.zip was missing from DRIVE_ZIPS={DRIVE_ZIPS}.'
    )
    assert len(_dbs) > 0, (
        f'Cannot run preprocess: 0 .db files at {LOCAL_LOGS}. '
        f'Either the extract+flatten cells did not run, or '
        f'nuplan-v1.1_mini.zip was missing from DRIVE_ZIPS={DRIVE_ZIPS}.'
    )
    print(f'Pre-preprocess OK: maps present, {len(_dbs)} .db files at {LOCAL_LOGS}')

# Fallback preprocess: train logs only via LOG_SPLIT_JSON's "train" key
# (54 of the 64 mini logs, deterministic given seed=42). Runs only if
# cell 22 could not load the Drive-cache shortcut. The training manifest
# is written by Section 7c, NOT here - this cell only produces the .npz
# files. Previous version wrote its own manifest, which clobbered the
# quarantined one Section 7c produces.
if not train_cache_ready:
    import os, pathlib
    os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
    os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
    os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

    # Ensure the log-disjoint split JSON exists. Deterministic given LOCAL_LOGS
    # contents + seed=42 - re-runs produce the same file byte-for-byte. If a
    # committed docs/log_split_mini_seed42.json already exists, this is a no-op.
    split_path = pathlib.Path(LOG_SPLIT_JSON)
    if not split_path.exists():
        print(f'Generating {split_path} from {LOCAL_LOGS}...')
        !{PYTHON} {REPO_DIR}/scripts/generate_log_split.py \
            --logs_dir {LOCAL_LOGS} \
            --output {split_path}

    # CONTRACT: only pass --total_scenarios when truthy (non-None, > 0).
    # The preprocess.py CLI default is None=unlimited; passing 0 or '' to
    # argparse converts to int(0) which some nuPlan versions interpret as
    # zero-scenarios (NOT unlimited). The cell 6 assertion already
    # rejects TOTAL_SCENARIOS=0, but explicitly omit the flag for clarity.
    _cap_flag = f'--total_scenarios {TOTAL_SCENARIOS}' if TOTAL_SCENARIOS else ''

    %cd {FP_DIR}
    !{PYTHON} -m flow_planner.run_script.preprocess \
        --data_path {LOCAL_LOGS} \
        --map_path {LOCAL_MAPS} \
        --save_path {LOCAL_CACHE} \
        {_cap_flag} \
        --log_names_json {LOG_SPLIT_JSON} \
        --log_names_key train \
        --seed {RUN_SEED}

    import pathlib
    npz_files = sorted(pathlib.Path(LOCAL_CACHE).glob('*.npz'))
    print(f'{len(npz_files)} .npz files in {LOCAL_CACHE} (manifest written by Section 7c)')

    # Post-preprocess count check (silent-truncation guard for the fallback
    # path; cell 27 only guards the Drive-cache path).
    if TOTAL_SCENARIOS is None:
        _floor = 5000
        assert len(npz_files) >= _floor, (
            f'Fallback preprocess produced only {len(npz_files)} files; '
            f'expected >= {_floor} for the full mini-split. '
            f'Check preprocess_failures.json in {LOCAL_CACHE} for per-scenario errors.'
        )
    else:
        if len(npz_files) == TOTAL_SCENARIOS:
            print(f'!!! NOTE: file count exactly equals TOTAL_SCENARIOS={TOTAL_SCENARIOS}. '
                  f'The safety cap was HIT - you may be silently truncating. '
                  f'Set TOTAL_SCENARIOS=None in cell 6 for the full mini-split.')
else:
    print('Drive cache loaded successfully in cell 22 - fallback preprocess not needed.')


In [ ]:
# diagnostic of where files actually are so the user can fix the path
# rather than guess what went wrong.
import pathlib as _p
_local = _p.Path(LOCAL_CACHE)
_npz_count = len(list(_local.glob('*.npz'))) if _local.exists() else 0
if _npz_count == 0:
    _drive_c = _p.Path(DRIVE_PREPROCESSED_1500)
    _drive_z = _p.Path(DRIVE_ZIPS)
    print('='*60)
    print('CACHE DIAGNOSTIC (training cache is empty)')
    print('='*60)
    print(f'LOCAL_CACHE             = {LOCAL_CACHE}')
    print(f'  exists                = {_local.exists()}')
    print(f'  .npz count            = {_npz_count}')
    print(f'DRIVE_PREPROCESSED_1500 = {DRIVE_PREPROCESSED_1500}')
    print(f'  exists                = {_drive_c.exists()}')
    if _drive_c.exists():
        print(f'  .npz count            = {len(list(_drive_c.glob("*.npz")))}')
    print(f'DRIVE_ZIPS              = {DRIVE_ZIPS}')
    print(f'  exists                = {_drive_z.exists()}')
    if _drive_z.exists():
        zips = sorted(_drive_z.glob('*.zip'))
        print(f'  .zip files            = {[z.name for z in zips]}')
    print(f'LOCAL_LOGS              = {LOCAL_LOGS}')
    _logs_p = _p.Path(LOCAL_LOGS)
    if _logs_p.exists():
        print(f'  .db count             = {len(list(_logs_p.glob("*.db")))}')
        nested_dbs = list(_logs_p.rglob('*.db'))
        if nested_dbs and not any(_logs_p.glob('*.db')):
            print(f'  NESTED .db files found at: {nested_dbs[0].parent}')
            print(f'  -> auto-flatten cell did not run or did not catch this layout')
    print(f'LOCAL_MAPS              = {LOCAL_MAPS}')
    _maps_p = _p.Path(LOCAL_MAPS)
    if _maps_p.exists():
        print(f'  has nuplan-maps-v1.0.json: {(_maps_p/"nuplan-maps-v1.0.json").exists()}')
        nested = list(_maps_p.rglob('nuplan-maps-v1.0.json'))
        if nested and not (_maps_p/"nuplan-maps-v1.0.json").exists():
            print(f'  NESTED maps json found at: {nested[0]}')
    print('='*60)
    print('Likely fixes:')
    print('  - If Drive zips are missing: upload nuplan-maps-v1.0.zip and')
    print('    nuplan-v1.1_mini.zip to', DRIVE_ZIPS)
    print('  - If files are at nested paths: re-run the auto-flatten cell')
    print('    that follows the extract cell, then re-run preprocess.')
    print('  - If you have a v6 Drive cache at a different path: update')
    print('    DRIVE_PREPROCESSED_1500 in the config cell.')
    print('='*60)
    raise RuntimeError(f'Training cache validation failed - 0 .npz files at {LOCAL_CACHE}. See diagnostic above.')

# Full per-file validation of the preprocessed cache. Implementation in
# scripts/validate_npz_cache.py — same script Section 8c uses for the
# held-out cache, so the validation rules can never drift between the
# two paths. Quarantines bad files from the training manifest; raises
# if more than 5% of files fail (systemic problem, not per-file accident).
import os, subprocess
r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/validate_npz_cache.py'],
    capture_output=True, text=True,
    env={**os.environ,
         'CACHE_DIR':       LOCAL_CACHE,
         'OUTPUT_MANIFEST': f'{LOCAL_CACHE}/diffusion_planner_training.json',
         'FAIL_THRESHOLD':  '0.05'},
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
    raise RuntimeError('Training cache validation failed — stop before training')


In [ ]:
# Set up raw nuPlan data + ensure the log-disjoint split JSON exists.
# This cell runs ALWAYS — Section 7's main path (Drive cache) bypasses
# unzipping for training, but the held-out path needs raw data either way.
import pathlib, zipfile

HELDOUT_AVAILABLE = False  # True only if BOTH raw data + log split are ready

# Explicit filename -> target mapping. Drop the substring-match approach
# (a future zip named "nuplan-with-maps-bundled.zip" would silently route
# to the wrong directory under substring routing).
ZIP_TARGETS = {
    'nuplan-maps-v1.0.zip': LOCAL_MAPS,
    'nuplan-v1.1_mini.zip': LOCAL_LOGS,
}

logs_dir = pathlib.Path(LOCAL_LOGS)
maps_dir = pathlib.Path(LOCAL_MAPS)
have_raw = logs_dir.exists() and any(logs_dir.iterdir()) and maps_dir.exists() and any(maps_dir.iterdir())

if have_raw:
    print('Raw nuPlan data already present locally — proceeding')
else:
    drive_zips = sorted(pathlib.Path(DRIVE_ZIPS).glob('*.zip'))
    print(f'Attempting to unzip {len(drive_zips)} zip(s) from Drive...')
    local_zip_dir = pathlib.Path(f'{LOCAL_ROOT}/_zips')
    local_zip_dir.mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_MAPS).mkdir(parents=True, exist_ok=True)
    pathlib.Path(LOCAL_LOGS).mkdir(parents=True, exist_ok=True)

    extracted_any = False
    for z in drive_zips:
        if z.name not in ZIP_TARGETS:
            print(f'  SKIP {z.name}: not in ZIP_TARGETS allowlist')
            continue
        target = ZIP_TARGETS[z.name]
        # If target is already populated (previous extraction completed),
        # skip the unzip step — saves several minutes on re-runs.
        if any(pathlib.Path(target).iterdir()):
            print(f'  {z.name}: {target} already populated, skipping extract')
            extracted_any = True
            continue
        dest = local_zip_dir / z.name
        if not dest.exists():
            !rsync --inplace {z} {dest}
        try:
            with zipfile.ZipFile(dest) as zf:
                print(f'  Unzipping {z.name} -> {target} ...')
                zf.extractall(target)
                extracted_any = True
        except zipfile.BadZipFile:
            # Continue with whatever other zips work rather than aborting
            # the whole held-out path on one bad zip.
            print(f'  CORRUPTED: {z.name} — skipping; held-out may be incomplete')

    have_raw = extracted_any and any(logs_dir.iterdir()) and any(maps_dir.iterdir())

if have_raw:
    # Generate the log split JSON if it isn't already present. Deterministic
    # given LOCAL_LOGS + seed=42 — re-runs produce the same file. If a
    # committed docs/log_split_mini_seed42.json already exists, this is a no-op.
    split_path = pathlib.Path(LOG_SPLIT_JSON)
    if not split_path.exists():
        print(f'\nGenerating log split {split_path} from {LOCAL_LOGS}...')
        !{PYTHON} {REPO_DIR}/scripts/generate_log_split.py \
            --logs_dir {LOCAL_LOGS} \
            --output {split_path}
    else:
        print(f'\nLog split already present: {split_path}')
    HELDOUT_AVAILABLE = True
    print('Raw data + log split ready — proceeding to held-out preprocess')
else:
    print('\nHeld-out skipped. Will use training-set ADE/FDE only in final table.')

# Preprocess HELD-OUT scenarios from the val logs only. The held-out cache
# draws from LOG_SPLIT_JSON's "val" key (10 of the 64 mini logs, disjoint
# from the "train" key cell 28 of Section 7b uses). This is the Phase 2
# methodology requirement - no log overlap with the training cache.
#
# The training manifest is written by cell 33 (Section 8c) AFTER per-file
# validation, NOT here. The previous version of this cell wrote its own
# all-files manifest, which would let bad files into the held-out eval.
import os, pathlib

if HELDOUT_AVAILABLE:
    existing = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
    expected = HELDOUT_SCENARIOS if (HELDOUT_SCENARIOS and HELDOUT_SCENARIOS > 0) else len(existing)

    if existing and len(existing) >= int(0.95 * expected):
        print(f'Held-out cache already populated ({len(existing)} files), skipping preprocess')
        print('(Section 8c will re-validate and write the manifest)')
    else:
        os.environ['NUPLAN_DATA_ROOT'] = LOCAL_NUPLAN
        os.environ['NUPLAN_MAPS_ROOT'] = LOCAL_MAPS
        os.environ['NUPLAN_EXP_ROOT']  = LOCAL_EXP

        # Mirror cell 32's CONTRACT: only pass --total_scenarios when truthy.
        _heldout_cap_flag = f'--total_scenarios {HELDOUT_SCENARIOS}' if HELDOUT_SCENARIOS else ''

        %cd {FP_DIR}
        !{PYTHON} -m flow_planner.run_script.preprocess \
            --data_path {LOCAL_LOGS} \
            --map_path {LOCAL_MAPS} \
            --save_path {LOCAL_HELDOUT_CACHE} \
            {_heldout_cap_flag} \
            --log_names_json {LOG_SPLIT_JSON} \
            --log_names_key val \
            --seed {RUN_SEED}

        npz_files = sorted(pathlib.Path(LOCAL_HELDOUT_CACHE).glob('*.npz'))
        print(f'\n{len(npz_files)} held-out .npz files (manifest written by Section 8c)')
        if len(npz_files) == 0:
            print('Preprocess returned 0 files - held-out skipped')
            HELDOUT_AVAILABLE = False
else:
    print('HELDOUT_AVAILABLE=False - skipping')

# produced no files, print a comprehensive diagnostic and bail. If
# HELDOUT_AVAILABLE is False (raw zips missing, etc.), this is a normal
# skip and we must NOT raise — the rest of the notebook treats held-out
# as best-effort (see Section 8 header and cell 70's guard).
import pathlib as _p
_local = _p.Path(LOCAL_HELDOUT_CACHE)
_npz_count = len(list(_local.glob('*.npz'))) if _local.exists() else 0
if HELDOUT_AVAILABLE and _npz_count == 0:
    _drive_c = _p.Path(DRIVE_PREPROCESSED_1500)
    _drive_z = _p.Path(DRIVE_ZIPS)
    print('='*60)
    print('CACHE DIAGNOSTIC (held-out cache is empty)')
    print('='*60)
    print(f'LOCAL_HELDOUT_CACHE     = {LOCAL_HELDOUT_CACHE}')
    print(f'  exists                = {_local.exists()}')
    print(f'  .npz count            = {_npz_count}')
    print(f'DRIVE_PREPROCESSED_1500 = {DRIVE_PREPROCESSED_1500}')
    print(f'  exists                = {_drive_c.exists()}')
    if _drive_c.exists():
        print(f'  .npz count            = {len(list(_drive_c.glob("*.npz")))}')
    print(f'DRIVE_ZIPS              = {DRIVE_ZIPS}')
    print(f'  exists                = {_drive_z.exists()}')
    if _drive_z.exists():
        zips = sorted(_drive_z.glob('*.zip'))
        print(f'  .zip files            = {[z.name for z in zips]}')
    print(f'LOCAL_LOGS              = {LOCAL_LOGS}')
    _logs_p = _p.Path(LOCAL_LOGS)
    if _logs_p.exists():
        print(f'  .db count             = {len(list(_logs_p.glob("*.db")))}')
        nested_dbs = list(_logs_p.rglob('*.db'))
        if nested_dbs and not any(_logs_p.glob('*.db')):
            print(f'  NESTED .db files found at: {nested_dbs[0].parent}')
            print(f'  -> auto-flatten cell did not run or did not catch this layout')
    print(f'LOCAL_MAPS              = {LOCAL_MAPS}')
    _maps_p = _p.Path(LOCAL_MAPS)
    if _maps_p.exists():
        print(f'  has nuplan-maps-v1.0.json: {(_maps_p/"nuplan-maps-v1.0.json").exists()}')
        nested = list(_maps_p.rglob('nuplan-maps-v1.0.json'))
        if nested and not (_maps_p/"nuplan-maps-v1.0.json").exists():
            print(f'  NESTED maps json found at: {nested[0]}')
    print('='*60)
    print('Likely fixes:')
    print('  - If Drive zips are missing: upload nuplan-maps-v1.0.zip and')
    print('    nuplan-v1.1_mini.zip to', DRIVE_ZIPS)
    print('  - If files are at nested paths: re-run the auto-flatten cell')
    print('    that follows the extract cell, then re-run preprocess.')
    print('  - If you have a v6 Drive cache at a different path: update')
    print('    DRIVE_PREPROCESSED_1500 in the config cell.')
    print('='*60)
    raise RuntimeError(f'Held-out cache validation failed - 0 .npz files at {LOCAL_HELDOUT_CACHE}. See diagnostic above.')
elif _npz_count == 0:
    print(f'HELDOUT_AVAILABLE=False and no held-out cache at {LOCAL_HELDOUT_CACHE} '
          '— skipping diagnostic (this is the normal best-effort skip path).')

import os, subprocess, json, pathlib

if HELDOUT_AVAILABLE:
    r = subprocess.run(
        [PYTHON, f'{REPO_DIR}/scripts/validate_npz_cache.py'],
        capture_output=True, text=True,
        env={**os.environ,
             'CACHE_DIR':       LOCAL_HELDOUT_CACHE,
             'OUTPUT_MANIFEST': f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
             'FAIL_THRESHOLD':  '0.05'},
    )
    print(r.stdout)
    if r.returncode != 0:
        print('STDERR:'); print(r.stderr)
        print('\nHeld-out validation failed — disabling held-out eval downstream')
        HELDOUT_AVAILABLE = False
    else:
        # Defense in depth: ensure no filename overlap between train and held-out
        # caches. The log-disjoint split guarantees this by construction; the
        # check exists so any methodology regression that bypasses the split
        # is caught.
        manifest_path = pathlib.Path(f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json')
        heldout_names = set(json.loads(manifest_path.read_text()))
        train_names = set(p.name for p in pathlib.Path(LOCAL_CACHE).glob('*.npz'))
        overlap = heldout_names & train_names
        if overlap:
            print(f'\n!!! UNEXPECTED train/heldout overlap: {len(overlap)} files')
            print('!!! The log-disjoint split should guarantee disjoint sets.')
            print('!!! Investigate before trusting this held-out cache.')
            print('!!! Removing overlapping files from manifest.')
            heldout_names -= overlap
            manifest_path.write_text(json.dumps(sorted(heldout_names)))
        print(f'\nHeld-out ready: {len(heldout_names)} scenarios (log-disjoint from train)')
else:
    print('HELDOUT_AVAILABLE=False — skipping held-out validation')


## 4. Train

Run training with `torchrun`. The override list is selected automatically from the `KINEMATIC` you set in Config. If Frenet is selected, normalization stats are re-measured from the current cache first. The resulting checkpoint is saved to `{DRIVE_FOLDER}/checkpoints/`.

In [ ]:
# Back up any existing checkpoint(s) to timestamped siblings before this
# section overwrites them. Defensive: makes regression bisection cheap.
#
# backed up if present, for safety on cross-version reruns).
import shutil, datetime, pathlib, glob
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Resolve V7_VARIANT / RUN_KINEMATIC defaults that match cell 57.
_v7_variant    = globals().get('V7_VARIANT', 'AUDIT')
_run_kinematic = globals().get('RUN_KINEMATIC', 'frenet')

candidates = [
    pathlib.Path(f'{DRIVE_CKPT}/v7{_v7_variant}_{_run_kinematic}_seed{RUN_SEED}.ckpt'),
    pathlib.Path(f'{DRIVE_CKPT}/{_run_kinematic}_seed{RUN_SEED}.ckpt'),
]
# variants still trigger a backup for whichever was last produced.
candidates += [pathlib.Path(p) for p in
               glob.glob(f'{DRIVE_CKPT}/v7*_{_run_kinematic}_seed{RUN_SEED}.ckpt')]

backed_up_any = False
seen = set()
for src in candidates:
    if str(src) in seen:
        continue
    seen.add(str(src))
    if src.exists():
        dst = src.with_name(f'{src.stem}_backup_{ts}.ckpt')
        shutil.copy(str(src), str(dst))
        print(f'Backed up {src.name} -> {dst.name}')
        backed_up_any = True

if not backed_up_any:
    print(f'No existing checkpoint at v7{_v7_variant}_{_run_kinematic}_seed{RUN_SEED}.ckpt '
          f'or legacy {_run_kinematic}_seed{RUN_SEED}.ckpt to back up (skipping)')


In [ ]:
# Implementation in scripts/measure_frenet_stats.py — runs in the venv,
# processes every .npz in LOCAL_CACHE (Section 7c already quarantined
# bad files into the manifest), and writes a JSON stats file consumed
# by cell 42 (Section 10.3) AND by cell 55 (Section 10.8 notes).
import os, subprocess
NEW_STATS_JSON = '/tmp/frenet_stats.json'
r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/measure_frenet_stats.py'],
    capture_output=True, text=True,
    env={**os.environ,
         'FP_DIR':      FP_DIR,
         'CACHE_DIR':   LOCAL_CACHE,
         'OUTPUT_JSON': NEW_STATS_JSON,
         'NUM_FILES':   '0'},  # 0 = process all files
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
    raise RuntimeError('Stats measurement failed')

# Lift the values into the cell namespace so later cells (10.3 YAML
# update, 10.8 notes) can reference them without re-reading the JSON.
import json
with open(NEW_STATS_JSON) as f:
    _stats = json.load(f)
NEW_MEAN = [
    round(_stats['s']['mean'], 2),
    round(_stats['d']['mean'], 2),
    _stats['cos_h_mean'],
    _stats['sin_h_mean'],
]
NEW_STD = [
    round(_stats['s']['std'], 2),
    round(_stats['d']['std'], 2),
    _stats['cos_h_std'],
    _stats['sin_h_std'],
]
print(f'NEW_MEAN = {NEW_MEAN}')
print(f'NEW_STD  = {NEW_STD}')

# Update flow_planner/script/normalization_stats/frenet_norm_stats.yaml
# from the JSON cell 40 (Section 10.2) wrote. Implementation in
# scripts/update_frenet_norm_stats.py — backs up the original YAML to
# .yaml.bak before overwriting and refuses to write if measured values
# are implausible (catches a botched measurement before it corrupts the
# YAML). Updates ego.uniform and neighbor.uniform blocks only; other
# blocks (ego_past, lanes, routes, etc.) are obs-side normalisation and
# stay untouched.
import os, subprocess
r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/update_frenet_norm_stats.py'],
    capture_output=True, text=True,
    env={**os.environ,
         'FP_DIR':     FP_DIR,
         'STATS_JSON': NEW_STATS_JSON},
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
    raise RuntimeError('YAML update failed')


In [ ]:
import os, pathlib, datetime, shutil

os.environ['PROJECT_ROOT']         = FP_DIR
os.environ['SAVE_DIR']             = LOCAL_RUNS
os.environ['TENSORBOARD_LOG_PATH'] = LOCAL_TB
os.environ['TRAINING_DATA']        = LOCAL_CACHE
os.environ['TRAINING_JSON']        = f'{LOCAL_CACHE}/diffusion_planner_training.json'
os.environ['WORLD_SIZE']           = '1'
os.environ['LOCAL_RANK']           = '0'
os.environ['MASTER_ADDR']          = 'localhost'
os.environ['MASTER_PORT']          = '29529'
os.environ['HYDRA_FULL_ERROR']     = '1'

# Sanity-check the training manifest before training. Section 7c
# quarantines bad .npz files; this assertion makes the dependency on a
# populated manifest explicit and fails loud if validation was skipped.
import json
_manifest = pathlib.Path(os.environ['TRAINING_JSON'])
assert _manifest.exists(), f'Training manifest {_manifest} missing - run Section 7c first.'
_files = json.loads(_manifest.read_text())
assert len(_files) >= 100, f'Training manifest has only {len(_files)} files; expected >= 100.'
print(f'Training on {len(_files)} validated scenarios')

# RUN_KINEMATIC and V7_VARIANT are read from cell 2 (Config) — the
# single source of truth for per-session knobs. Do NOT redefine here.
assert 'RUN_KINEMATIC' in globals(), 'RUN_KINEMATIC must be set in cell 2 (Config)'
assert 'V7_VARIANT' in globals(),    'V7_VARIANT must be set in cell 2 (Config)'
assert V7_VARIANT in {'AUDIT','h','b','c','d'}, (
    f'V7_VARIANT={V7_VARIANT!r} not in {{AUDIT,h,b,c,d}}'
)
assert RUN_KINEMATIC in {'frenet','waypoints'}, (
    f'RUN_KINEMATIC={RUN_KINEMATIC!r} not in {{frenet,waypoints}}'
)

if V7_VARIANT == 'h':
    if len(_files) < 5000:
        print(f'!!! v7h WARNING: requested scale-up but training manifest only has '
              f'{len(_files)} scenarios - set TOTAL_SCENARIOS=None in cell 6 and re-run '
              f'Section 7 to actually scale up. Reporting actual scenarios={len(_files)} '
              f'in results.csv (no longer the misleading 999999 label).')
    TOTAL_SCENARIOS = len(_files)
    EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE
    print(f'v7h: TOTAL_SCENARIOS pinned to manifest size ({TOTAL_SCENARIOS})')
else:
    # Loud warning if the manifest size disagrees with the cell-6 cap.
    if TOTAL_SCENARIOS is not None and len(_files) != TOTAL_SCENARIOS:
        slack = max(1, int(0.05 * TOTAL_SCENARIOS))
        if abs(len(_files) - TOTAL_SCENARIOS) > slack:
            print(f'!!! WARNING: training manifest has {len(_files)} files but '
                  f'TOTAL_SCENARIOS={TOTAL_SCENARIOS}. results.csv will report the '
                  f'cell-6 cap, NOT the manifest size. Investigate before publishing.')

NORM_STATS    = 'frenet_norm_stats' if RUN_KINEMATIC == 'frenet' else 'waypoints_norm_stats'
RUN_NAME      = f'v7{V7_VARIANT}_{RUN_KINEMATIC}_seed{RUN_SEED}'
print(f'v7 run: variant={V7_VARIANT}, kinematic={RUN_KINEMATIC}, seed={RUN_SEED}, run_name={RUN_NAME}')

# Archive any previous run's log instead of wiping it - keeps a paper
# trail of failed attempts. Then clear LOCAL_RUNS for a clean training.
prev_log = pathlib.Path(LOCAL_RUNS) / f'{RUN_NAME}.log'
if prev_log.exists():
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    archive_dir = pathlib.Path(f'{LOCAL_ROOT}/runs_archive')
    archive_dir.mkdir(parents=True, exist_ok=True)
    archived = archive_dir / f'{RUN_NAME}_{ts}.log'
    shutil.move(str(prev_log), str(archived))
    print(f'Archived previous log to {archived}')

# Use shutil.rmtree (no ignore_errors) so a real cleanup failure halts the
# run instead of silently letting stale checkpoints from a prior run
# pollute the next sort. The shell `!rm -rf LOCAL_RUNS/*` magic was even
# worse - `!` swallows non-zero exit codes outright.
runs_dir = pathlib.Path(LOCAL_RUNS)
if runs_dir.exists():
    shutil.rmtree(runs_dir, ignore_errors=False)
runs_dir.mkdir(parents=True, exist_ok=True)
print(f'Cleared {runs_dir} for fresh training run')

import subprocess, pathlib
%cd {FP_DIR}

CENTERLINE_OVERRIDE = (
    ' +model.model_encoder.centerline_encoder._target_=flow_planner.model.modules.encoder_modules.CenterlineEncoder'
    ' +model.model_encoder.centerline_encoder.n_points=100'
    ' +model.model_encoder.centerline_encoder.hidden_dim=256'
    ' ++model.model_decoder.enable_attn_dist=false'
) if RUN_KINEMATIC == 'frenet' else ''

cmd = (
    f"{VENV}/bin/torchrun --nnodes 1 --nproc-per-node 1 --standalone "
    f"flow_planner/trainer.py --config-name flow_planner_standard "
    f"model.kinematic={RUN_KINEMATIC} "
    f"normalization_stats={NORM_STATS} "
    f"train.batch_size={EFFECTIVE_BATCH_SIZE} "
    f"train.epoch={TRAIN_EPOCHS} "
    f"scheduler.warm_up_epoch={WARM_UP_EPOCHS} "
    f"train.save_utd=1 "
    f"save_every_since={TRAIN_EPOCHS} "
    f"ddp.distributed=false "
    f"seed={RUN_SEED} "
    f"job_name={RUN_NAME} "
    f"num_workers=2"
    f"{CENTERLINE_OVERRIDE}"
)
log_path = pathlib.Path(LOCAL_RUNS) / f'{RUN_NAME}.log'
print(f'Launching training; logs at {log_path}')

# Stream output to the log file (no tee - tqdm progress flooding the
# notebook would crash the browser tab on long runs). Capture the exit
# code so downstream cells halt on training failure.
with open(log_path, 'w') as logf:
    r = subprocess.run(['bash', '-c', cmd], stdout=logf, stderr=subprocess.STDOUT)

print(f"\n=== last 50 lines of {log_path} ===")
!tail -50 {log_path}

assert r.returncode == 0, (
    f"Training failed (exit code {r.returncode}). "
    f"See {log_path} for full output."
)


In [ ]:
import pathlib, shutil
ckpts = list(pathlib.Path(LOCAL_RUNS).rglob('*.ckpt')) + list(pathlib.Path(LOCAL_RUNS).rglob('*.pth'))
assert ckpts, f'No checkpoint found under {LOCAL_RUNS}'
best_ckpt = sorted(ckpts, key=lambda p: p.stat().st_mtime)[-1]
print(f'Most recent: {best_ckpt}')

pathlib.Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
ckpt_drive_path = f'{DRIVE_CKPT}/{RUN_NAME}.ckpt'

# Use shutil.copy (raises on failure) and assert the destination size
# matches the source. `!cp` is a shell magic — `!` swallows non-zero exit
# codes, so a Drive-FUSE blip / quota error / read-only remount would
# silently leave ckpt_drive_path pointing at the PRIOR run's checkpoint
# (same RUN_NAME path), and downstream eval would report numbers from
# the wrong weights.
shutil.copy(str(best_ckpt), ckpt_drive_path)
src_size = best_ckpt.stat().st_size
dst_size = pathlib.Path(ckpt_drive_path).stat().st_size
assert src_size == dst_size, (
    f'Drive copy size mismatch: src={src_size} dst={dst_size}. '
    f'Refusing to proceed — downstream eval would report numbers from a '
    f'partial / stale checkpoint.'
)
print(f'Saved to {ckpt_drive_path} ({dst_size:,} bytes)')


## 5. Eval

Eval the trained checkpoint on the **training-set** scenes and parse ADE/FDE/miss-rate. If a held-out cache was built in Section 3, the same eval runs on **held-out** scenes for the current kinematic (and for the matching baseline if its checkpoint is already on Drive).

In [ ]:
import subprocess
eval_train_json = f'{LOCAL_ROOT}/eval_train.json'
%cd {FP_DIR}

# --num_batches 999 effectively evaluates on the entire training cache.
# The previous version capped at 10 batches (320 scenarios of ~1500),
# so the reported ADE was on a 21% sample.
r = subprocess.run([
    PYTHON, '-m', 'flow_planner.run_script.inference_eval',
    '--checkpoint',  ckpt_drive_path,
    '--data_dir',    LOCAL_CACHE,
    '--data_list',   f'{LOCAL_CACHE}/diffusion_planner_training.json',
    '--kinematic',   RUN_KINEMATIC,
    '--norm_stats',  NORM_STATS,
    '--output_json', eval_train_json,
    '--batch_size',  str(EFFECTIVE_BATCH_SIZE),
    '--num_batches', '999',
], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
assert r.returncode == 0, f'Inference eval failed (returncode={r.returncode})'

import json
with open(eval_train_json) as f:
    train_eval = json.load(f)
print(f'{RUN_KINEMATIC.title()} on TRAINING set:')
for k, v in train_eval.items():
    print(f'  {k}: {v}')


In [ ]:
if not HELDOUT_AVAILABLE:
    print('Held-out eval skipped: HELDOUT_AVAILABLE is False (no held-out cache built in Section 3).')

current_heldout   = None
waypoints_heldout = None

if HELDOUT_AVAILABLE:
    import subprocess, json
    eval_heldout_current_json = f'{LOCAL_ROOT}/eval_heldout_current.json'
    %cd {FP_DIR}
    r = subprocess.run([
        PYTHON, '-m', 'flow_planner.run_script.inference_eval',
        '--checkpoint',  ckpt_drive_path,
        '--data_dir',    LOCAL_HELDOUT_CACHE,
        '--data_list',   f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
        '--kinematic',   RUN_KINEMATIC,
        '--norm_stats',  NORM_STATS,
        '--output_json', eval_heldout_current_json,
        '--batch_size',  str(EFFECTIVE_BATCH_SIZE),
        '--num_batches', '999',
        '--seed',        str(RUN_SEED),
    ], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print('STDERR:'); print(r.stderr)
    assert r.returncode == 0, f'{RUN_KINEMATIC} held-out eval failed'

    with open(eval_heldout_current_json) as f:
        current_heldout = json.load(f)
    # Back-compat alias so any external cells still reading frenet_heldout
    # do not break.
    frenet_heldout = current_heldout
    print(f'\n{RUN_KINEMATIC.title()} on HELD-OUT:')
    for k, v in current_heldout.items():
        print(f'  {k}: {v}')

# Waypoints on held-out cache (compare against Frenet under same held-out
# cache so the ADE/FDE numbers are directly comparable). Skips cleanly if
# no waypoints checkpoint is present.
#
# Lookup order matches cell 60's save naming convention:
#   1. v7{V7_VARIANT}_waypoints_seed{RUN_SEED}.ckpt   (current v7 runs)
#   2. waypoints_seed{RUN_SEED}.ckpt                  (legacy v6 runs)
#   3. glob v7*_waypoints_seed{RUN_SEED}.ckpt          (any v7 variant)
if HELDOUT_AVAILABLE:
    import subprocess, json, pathlib, glob
    primary_name = f'v7{V7_VARIANT}_waypoints_seed{RUN_SEED}.ckpt'
    legacy_name  = f'waypoints_seed{RUN_SEED}.ckpt'

    candidate_paths = [
        f'{DRIVE_CKPT}/{primary_name}',
        f'{DRIVE_CKPT}/{legacy_name}',
    ]
    # Glob fallback: any v7 variant at the same seed.
    candidate_paths += sorted(
        glob.glob(f'{DRIVE_CKPT}/v7*_waypoints_seed{RUN_SEED}.ckpt'),
        key=lambda p: pathlib.Path(p).stat().st_mtime if pathlib.Path(p).exists() else 0,
        reverse=True,
    )

    waypoints_ckpt = None
    for cand in candidate_paths:
        if pathlib.Path(cand).exists():
            waypoints_ckpt = cand
            break

    if waypoints_ckpt is None:
        print(f'No waypoints checkpoint found at any of:')
        for cand in candidate_paths:
            print(f'  - {cand}')
        print('Skipping comparison.')
        waypoints_heldout = None
    else:
        print(f'Using waypoints checkpoint: {waypoints_ckpt}')
        eval_heldout_wp_json = f'{LOCAL_ROOT}/eval_heldout_waypoints.json'
        WP_RUN_NAME = pathlib.Path(waypoints_ckpt).stem
        %cd {FP_DIR}
        r = subprocess.run([
            PYTHON, '-m', 'flow_planner.run_script.inference_eval',
            '--checkpoint',  waypoints_ckpt,
            '--data_dir',    LOCAL_HELDOUT_CACHE,
            '--data_list',   f'{LOCAL_HELDOUT_CACHE}/diffusion_planner_training.json',
            '--kinematic',   'waypoints',
            '--norm_stats',  'waypoints_norm_stats',
            '--output_json', eval_heldout_wp_json,
            '--batch_size',  str(EFFECTIVE_BATCH_SIZE),
            '--num_batches', '999',
            '--seed',        str(RUN_SEED),
        ], capture_output=True, text=True)
        print(r.stdout)
        if r.returncode != 0:
            print('STDERR:'); print(r.stderr)
        assert r.returncode == 0, f'Waypoints held-out eval failed (returncode={r.returncode})'

        with open(eval_heldout_wp_json) as f:
            waypoints_heldout = json.load(f)
        print('\nWaypoints on HELD-OUT:')
        for k, v in waypoints_heldout.items():
            print(f'  {k}: {v}')


## 6. Save

Generate BEV visualizations (GT vs prediction with centerline overlay), append a row per (kinematic, split) to the shared `results.csv` on Drive, then display the latest comparison table and all BEV PNGs from this run.

In [ ]:
# BEV visualisation: GT vs prediction for 4 scenes.
# Implementation lives at scripts/visualize_bev.py — runs in the venv
# (needs Hydra + the Flow Planner model classes).
# Pass KINEMATIC + NORM_STATS so the script can branch correctly when
# we're evaluating a waypoints checkpoint (it shouldn't apply the
# Frenet-only centerline encoder overrides or the frenet_to_cartesian
# decode — those would produce meaningless trajectories from waypoint
# (x,y) predictions interpreted as Frenet (s,d)).
import os, subprocess, pathlib

bev_png = f'{LOCAL_ROOT}/bev_{RUN_NAME}.png'
# Belt-and-braces: cell 57 already unlinks, but if a user runs cells out
# of order, this guarantees a stale PNG never masquerades as fresh output.
pathlib.Path(bev_png).unlink(missing_ok=True)

r = subprocess.run(
    [PYTHON, f'{REPO_DIR}/scripts/visualize_bev.py'],
    capture_output=True, text=True,
    env={
        **os.environ,
        'FP_DIR':     FP_DIR,
        'CKPT_PATH':  ckpt_drive_path,
        'CACHE_DIR':  LOCAL_CACHE,
        'OUTPUT_PNG': bev_png,
        'PLOT_TITLE': RUN_NAME,
        'KINEMATIC':  RUN_KINEMATIC,
        'NORM_STATS': NORM_STATS,
    },
)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:'); print(r.stderr)
assert r.returncode == 0, (
    f'BEV viz failed (exit code {r.returncode}). See STDERR above. '
    f'Stopping rather than letting cell 66 display a stale PNG.'
)

from IPython.display import Image, display
import pathlib
if pathlib.Path(bev_png).exists():
    display(Image(bev_png))
else:
    print(f'BEV PNG not found at {bev_png} — skipping display.')
    print('(Cell 65 should have failed loud if visualize_bev.py errored; '
          'check that cell\'s output for the underlying cause.)')


In [ ]:
import shutil, pathlib, datetime, csv, json

exp_id = f'{RUN_NAME}_scen{TOTAL_SCENARIOS}_ep{TRAIN_EPOCHS}'
artifact_dir = pathlib.Path(f'{DRIVE_FOLDER}/experiments/{exp_id}')
artifact_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(bev_png, artifact_dir / 'bev.png')
shutil.copy(eval_train_json, artifact_dir / 'eval_train.json')
shutil.copy(f'{FP_DIR}/flow_planner/script/normalization_stats/frenet_norm_stats.yaml',
            artifact_dir / 'frenet_norm_stats.yaml')

results_csv = f'{DRIVE_RESULTS}/results.csv'
pathlib.Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)
prior = []
if pathlib.Path(results_csv).exists():
    with open(results_csv) as f:
        prior = list(csv.DictReader(f))

comparison_lines = []
for r in prior[-5:]:
    name  = r.get('run_name', '<no-name>')
    kin   = r.get('kinematic', '?')
    split = r.get('split', '?')
    try:
        ade_m = float(r.get('ade_mean'))
        ade_s = float(r.get('ade_std'))
    except (TypeError, ValueError):
        comparison_lines.append(f"  {name} ({kin}, {split}): <legacy row, skipped>")
        continue
    comparison_lines.append(f"  {name} ({kin}, {split}): ADE={ade_m:.2f} +- {ade_s:.2f} m")
comparison = chr(10).join(comparison_lines) if comparison_lines else '  (no readable prior runs)'

ade_mean = train_eval["ade_mean"]; ade_std = train_eval["ade_std"]
fde_mean = train_eval["fde_mean"]; fde_std = train_eval["fde_std"]
now = datetime.datetime.now().isoformat(timespec='seconds')

notes = (
    f'Experiment: {exp_id}\nDate: {now}\n\n'
    f'Config: kinematic={RUN_KINEMATIC}, scenarios={TOTAL_SCENARIOS}, '
    f'epochs={TRAIN_EPOCHS}, batch={EFFECTIVE_BATCH_SIZE}, seed={RUN_SEED}\n'
    f'Norm stats: mean={NEW_MEAN}, std={NEW_STD}\n\n'
    f'Train-set ADE: {ade_mean:.3f} +- {ade_std:.3f} m\n'
    f'Train-set FDE: {fde_mean:.3f} +- {fde_std:.3f} m\n\n'
    f'Last 5 runs in results.csv:\n{comparison}\n'
)
(artifact_dir / 'NOTES.md').write_text(notes)
print(f'Artifacts -> {artifact_dir}')

row = {
    'run_name': RUN_NAME, 'kinematic': RUN_KINEMATIC, 'seed': RUN_SEED,
    'scenarios': TOTAL_SCENARIOS, 'epochs': TRAIN_EPOCHS,
    'batch_size': EFFECTIVE_BATCH_SIZE, 'split': 'train',
    'ade_mean': ade_mean, 'ade_std': ade_std,
    'fde_mean': fde_mean, 'fde_std': fde_std,
    'timestamp': now,
}
exists = pathlib.Path(results_csv).exists()
with open(results_csv, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=row.keys())
    if not exists:
        w.writeheader()
    w.writerow(row)
print(f'Appended to {results_csv}')

if HELDOUT_AVAILABLE:
    import csv, datetime, pathlib
    results_csv = f'{DRIVE_RESULTS}/results.csv'
    pathlib.Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)

    rows_to_save = []
    # Current-run row (whatever kinematic this notebook session is on).
    if 'current_heldout' in dir() and current_heldout is not None:
        rows_to_save.append((RUN_KINEMATIC, RUN_NAME, current_heldout))
    elif 'frenet_heldout' in dir() and frenet_heldout is not None:
        # Back-compat with older notebook state that only defined frenet_heldout.
        rows_to_save.append((RUN_KINEMATIC, RUN_NAME, frenet_heldout))
    # Comparison row: skip when this run IS the waypoints run (would
    # duplicate the row just added above).
    if (
        RUN_KINEMATIC != 'waypoints'
        and 'waypoints_heldout' in dir()
        and waypoints_heldout is not None
    ):
        rows_to_save.append(('waypoints', f'waypoints_seed{RUN_SEED}', waypoints_heldout))

    # Loud guard against silently writing two rows with the same
    # (kinematic, run_name) key.
    keys = [(r[0], r[1]) for r in rows_to_save]
    assert len(set(keys)) == len(keys), (
        f'Refusing to write duplicate (kinematic, run_name) rows to results.csv: {keys}'
    )

    for rep, name, result in rows_to_save:
        row = {
            'run_name': name, 'kinematic': rep, 'seed': RUN_SEED,
            'scenarios': TOTAL_SCENARIOS, 'epochs': TRAIN_EPOCHS,
            'batch_size': EFFECTIVE_BATCH_SIZE, 'split': 'heldout',
            'ade_mean': result['ade_mean'], 'ade_std': result['ade_std'],
            'fde_mean': result['fde_mean'], 'fde_std': result['fde_std'],
            'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
        }
        exists = pathlib.Path(results_csv).exists()
        with open(results_csv, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=row.keys())
            if not exists:
                w.writeheader()
            w.writerow(row)
        print(f'  appended {rep} held-out: ADE={row["ade_mean"]:.3f} m')


In [ ]:
# Read results.csv and display the latest row per (kinematic, split).
# Also writes the same table to a Drive artifact so it can be lifted
# into the report without re-running this cell.
#
# Robust against schema drift: if the CSV header is from a pre-cleanup
# run (no 'split' column), we migrate it in place to the canonical
# 12-field schema. Legacy rows (11 fields) get split='train' by
# convention; new rows (12 fields written by Section 10.8 / Section 11)
# are re-aligned to the canonical order.
import csv, pathlib

CANONICAL_FIELDS = [
    'run_name', 'kinematic', 'seed', 'scenarios', 'epochs', 'batch_size',
    'split', 'ade_mean', 'ade_std', 'fde_mean', 'fde_std', 'timestamp',
]

results_csv = f'{DRIVE_RESULTS}/results.csv'
# Mirror cells 26/68/73 — create the parent dir if missing so this cell
# never raises FileNotFoundError on a fresh-Drive setup.
pathlib.Path(DRIVE_RESULTS).mkdir(parents=True, exist_ok=True)

if not pathlib.Path(results_csv).exists():
    print(f'{results_csv} does not exist yet — run Section 10.8 (cell 68) '
          'and/or Section 11 (cell 73) first to populate results.')
    rows = []
else:
    # Migrate the CSV to canonical schema if needed.
    with open(results_csv) as f:
        raw = list(csv.reader(f))

    if not raw:
        print('results.csv is empty')
        rows = []
    else:
        header, *data = raw
        if header != CANONICAL_FIELDS:
            print(f'Detected old CSV schema ({len(header)} fields). Migrating to canonical.')
            migrated = []
            for row in data:
                if len(row) == 11:
                    # Legacy row: insert 'train' for the missing split column at position 6.
                    migrated.append(row[:6] + ['train'] + row[6:])
                elif len(row) == 12:
                    # New row already in canonical order, just needs the canonical header above it.
                    migrated.append(row)
                else:
                    print(f'  SKIP malformed row of length {len(row)}: {row[:4]}...')
            with open(results_csv, 'w', newline='') as f:
                w = csv.writer(f)
                w.writerow(CANONICAL_FIELDS)
                w.writerows(migrated)
            print(f'  Migrated {len(migrated)} rows to canonical schema')

        with open(results_csv) as f:
            rows = list(csv.DictReader(f))

# Latest row per (kinematic, split), tie-break by timestamp.
latest = {}
for r in rows:
    try:
        # Defensive: skip rows where ade_mean can't be parsed (corrupt).
        float(r['ade_mean'])
    except (KeyError, ValueError, TypeError):
        continue
    key = (r.get('kinematic', '?'), r.get('split', 'train'))
    if key not in latest or r.get('timestamp', '') > latest[key].get('timestamp', ''):
        latest[key] = r

header = f'{"Rep":<12} {"Split":<10} {"ADE (m)":<20} {"FDE (m)":<20} {"Run":<40}'
separator = '-' * len(header)
lines = [header, separator]
for (rep, split), r in sorted(latest.items()):
    ade = f'{float(r["ade_mean"]):.2f} +/- {float(r["ade_std"]):.2f}'
    fde = f'{float(r["fde_mean"]):.2f} +/- {float(r["fde_std"]):.2f}'
    lines.append(f'{rep:<12} {split:<10} {ade:<20} {fde:<20} {r["run_name"]:<40}')

table = chr(10).join(lines)
print(table)

# Save the table as an artifact so the report writer can use it directly.
artifact = pathlib.Path(f'{DRIVE_FOLDER}/experiments/_latest_comparison.txt')
artifact.parent.mkdir(parents=True, exist_ok=True)
artifact.write_text(table + chr(10))
print(f'{chr(10)}Saved table to {artifact}')

# Display every BEV PNG we wrote during this notebook run. Names follow
# the pattern bev_{RUN_NAME}.png (per cell 52) so we glob for them.
from IPython.display import Image, display
import pathlib

bev_paths = sorted(pathlib.Path(LOCAL_ROOT).glob('bev_*.png'))
if not bev_paths:
    print(f'No BEV PNGs found in {LOCAL_ROOT}/')
else:
    for p in bev_paths:
        label = p.stem.replace('bev_', '').replace('_', ' ')
        print(f'=== {label} ===')
        display(Image(str(p)))

print(f"Artifacts saved to {DRIVE_FOLDER}")
